<a href="https://colab.research.google.com/github/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-/blob/main/notebooks/cross_dataset_alert_budgets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alert budgets across datasets

PR-AUC summarises the whole ranking, but a bank only reviews a small share of its transactions. In `final_model_gap.ipynb` we measured on Sparkov how much fraud the model catches when analysts review only the top 0.5%, 1% or 2% of test transactions. In this notebook we measure the same alert budgets for every feature group in `cross_dataset_protocol.ipynb`, on every dataset.

For each budget we report:
- **Recall:** the share of fraud transactions that fall inside the budget;
- **Dollar recall:** the share of fraud dollars that fall inside the budget (refunds and negative amounts count as 0);
- **Precision:** the share of reviewed transactions that are fraud.

A model that ranks at random catches about the same share of fraud as the budget itself, so its recall at 1% is about 0.01. Recall is easier to compare across datasets than precision, because precision depends on the fraud rate.

We use exactly the same data, features, split and models as `cross_dataset_protocol.ipynb`. The predictions are loaded from the seeds that notebook saved to Google Drive, so nothing is retrained. The 95% intervals come from the same day-block bootstrap (500 resamples, `BOOT_SEED = 2026`), and each number is the average over the five seeds.

The notebook runs the three datasets one after another and shows the results of each in Section 6. Each dataset's results are saved to Google Drive, and Section 7 combines them into Table 3.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook. We use the same `QUICK` setting as in `cross_dataset_protocol.ipynb`, so the saved seeds are found.

In [1]:
# 1.1 Settings
DATASETS = ['cc2026', 'sparkov', 'tabformer']   # run in this order; remove one to skip it
QUICK    = False                                 # True = 1 seed, 100 trees, 200 bootstrap resamples (a fast preview only)

SEEDS     = [42] if QUICK else [42, 43, 44, 45, 46]
N_TREES   = 100 if QUICK else 200
N_BOOT    = 200 if QUICK else 500
BOOT_SEED = 2026

# TabFormer: the same user sample and window as tabformer_validation.ipynb
TF_N_USERS, TF_SAMPLE_SEED, TF_WINDOW_START = 400, 42, '2015-01-01'

KAGGLE_SLUGS = {
    'sparkov':   'kartik2112/fraud-detection',
    'tabformer': 'ealtman2019/credit-card-transactions',
    'cc2026':    'mohankrishnathalla/credit-card-fraud-detection-dataset-2026',
}
NAMES = {'sparkov': 'Sparkov', 'tabformer': 'TabFormer', 'cc2026': 'CC Fraud 2026'}
assert all(ds in KAGGLE_SLUGS for ds in DATASETS)
print(f'datasets: {DATASETS} | seeds: {SEEDS} | trees: {N_TREES} | bootstrap: {N_BOOT}')

datasets: ['cc2026', 'sparkov', 'tabformer'] | seeds: [42, 43, 44, 45, 46] | trees: 200 | bootstrap: 500


In [2]:
# 1.2 Download the datasets from Kaggle
import os, glob
from getpass import getpass

def data_dir(ds):
    return f'/content/data/{ds}'

for ds in DATASETS:
    if not glob.glob(f'{data_dir(ds)}/**/*.csv', recursive=True):
        if 'KAGGLE_KEY' not in os.environ:
            os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
            os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')
        !kaggle datasets download -d {KAGGLE_SLUGS[ds]} -p {data_dir(ds)} --unzip -q
    for f in sorted(glob.glob(f'{data_dir(ds)}/**/*', recursive=True)):
        if os.path.isfile(f):
            print(f'{os.path.getsize(f) / 1e6:9.1f} MB  {f}')

Kaggle username: Subbareddy_subbz
Kaggle API key: ··········
Dataset URL: https://www.kaggle.com/datasets/mohankrishnathalla/credit-card-fraud-detection-dataset-2026
License(s): CC0-1.0
     25.7 MB  /content/data/cc2026/credit_card_fraud_detection_2026.csv
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
    150.4 MB  /content/data/sparkov/fraudTest.csv
    351.2 MB  /content/data/sparkov/fraudTrain.csv
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/credit-card-transactions
License(s): other
      1.9 MB  /content/data/tabformer/User0_credit_card_transactions.csv
   2350.7 MB  /content/data/tabformer/credit_card_transactions-ibm_v2.csv
      0.5 MB  /content/data/tabformer/sd254_cards.csv
      0.2 MB  /content/data/tabformer/sd254_users.csv


In [3]:
# 1.3 Clone the project repository to use our helpers
REPO = 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'
if not os.path.exists('src/evaluation.py'):
    if not os.path.exists(REPO):
        github_token = getpass('Paste your GitHub token: ').strip()
        !git clone https://{github_token}@github.com/subbareddypullalacheruvu/{REPO}.git
    %cd {REPO}

Paste your GitHub token: ··········
Cloning into 'Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-'...
remote: Enumerating objects: 369, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (176/176), done.
remote: Total 369 (delta 147), reused 53 (delta 44), pack-reused 149 (from 1)
Receiving objects: 100% (369/369), 11.97 MiB | 27.61 MiB/s, done.
Resolving deltas: 100% (225/225), done.
/content/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-


In [4]:
# 1.4 Imports and the Google Drive folders for progress and results
import sys, gc
import numpy as np
from IPython.display import display
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.getcwd())
from src.utils import fit_category_risk, apply_category_risk, haversine_distance
from src.evaluation import chronological_val_split, run_seeds, bootstrap_model, summarise, format_ci
pd.set_option('display.max_columns', 60)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol'
RESULTS_DIR = f'{BASE}/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

def checkpoint_dir(ds):
    path = f"{BASE}/{ds}{'_quick' if QUICK else ''}"
    os.makedirs(path, exist_ok=True)
    return path

for ds in DATASETS:
    print(f'progress ({ds}) ->', checkpoint_dir(ds))
print('results ->', RESULTS_DIR)

Mounted at /content/drive
progress (cc2026) -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/cc2026
progress (sparkov) -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/sparkov
progress (tabformer) -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/tabformer
results -> /content/drive/MyDrive/fraud_checkpoints/cross_dataset_protocol/results


## 2. Loading the data, building the features and splitting by time
These cells are the same as Sections 2 to 6 of `cross_dataset_protocol.ipynb`, so the test rows are in the same order as the saved predictions. They only define the steps; Section 6 runs them for each dataset.

In [5]:
# 2.1 One loader per dataset
def csv_files(ds):
    return glob.glob(f'{data_dir(ds)}/**/*.csv', recursive=True)

def load_sparkov():
    files = csv_files('sparkov')
    tr = pd.read_csv(next(f for f in files if f.endswith('fraudTrain.csv')))
    te = pd.read_csv(next(f for f in files if f.endswith('fraudTest.csv')))
    tr['_split'], te['_split'] = 'train', 'test'          # Kaggle's own train/test files, as in all our notebooks
    d = pd.concat([tr, te], ignore_index=True)
    d = d.drop(columns=[c for c in d.columns if c.startswith('Unnamed')])
    d['_time'] = pd.to_datetime(d['trans_date_trans_time'])
    d['_y'] = d['is_fraud'].astype(int)
    d['_card'] = d['cc_num'].astype(str)
    d['_amount'] = d['amt'].astype(float)
    d['_category'] = d['category'].astype(str)
    d['age'] = (d['_time'] - pd.to_datetime(d['dob'])).dt.days / 365.25
    d['gender_flag'] = (d['gender'] == 'M').astype(int)
    skip = {'trans_date_trans_time', 'cc_num', 'first', 'last', 'street', 'trans_num', 'unix_time',
            'dob', 'is_fraud', '_split'}
    native = [c for c in tr.columns if c not in skip and not c.startswith('Unnamed')]
    text = {'category', 'gender', 'city', 'state', 'job', 'merchant', 'zip'}
    return d, native, text

def load_tabformer():
    files = csv_files('tabformer')
    users_csv = next(f for f in files if 'user' in os.path.basename(f).lower()
                     and 'transaction' not in os.path.basename(f).lower())
    txn_csv = max((f for f in files if 'transaction' in os.path.basename(f).lower()), key=os.path.getsize)
    users = pd.read_csv(users_csv)
    users['User'] = np.arange(len(users))               # transactions refer to users by row number
    rng = np.random.default_rng(TF_SAMPLE_SEED)
    sampled = set(rng.choice(users['User'].values, size=min(TF_N_USERS, len(users)), replace=False).tolist())
    parts = [ch[ch['User'].isin(sampled)] for ch in pd.read_csv(txn_csv, chunksize=2_000_000, low_memory=False)]
    raw = pd.concat(parts, ignore_index=True)
    txn_cols = list(raw.columns)

    d = raw.copy()
    d['_time'] = pd.to_datetime(dict(year=d['Year'], month=d['Month'], day=d['Day']), errors='coerce') \
                 + pd.to_timedelta(d['Time'].astype(str) + ':00', errors='coerce')
    d['_y'] = d['Is Fraud?'].astype(str).str.strip().str.lower().isin(['yes', '1', 'true']).astype(int)
    d['_amount'] = pd.to_numeric(d['Amount'].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
    d['_card'] = d['User'].astype(str) + '_' + d['Card'].astype(str)     # one card = user + card number
    d['_category'] = d['MCC'].astype(str)
    user_cols = [c for c in users.columns if c not in d.columns]
    d = d.merge(users[['User'] + user_cols], on='User', how='left')
    birth = pd.to_datetime(dict(year=d['Birth Year'], month=d['Birth Month'], day=1), errors='coerce')
    d['age'] = (d['_time'] - birth).dt.days / 365.25
    d['gender_flag'] = d['Gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])

    skip = {'User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Is Fraud?', 'Merchant Name', 'Zip',
            'Person', 'Address', 'Apartment', 'Zipcode', 'Birth Month'}
    native = [c for c in txn_cols + user_cols if c not in skip]
    text = {'MCC', 'Merchant State', 'Merchant City', 'State', 'Gender', 'City', 'Use Chip', 'Errors?'}
    return d, native, text

def load_cc2026():
    d = pd.read_csv(max(csv_files('cc2026'), key=os.path.getsize), low_memory=False)
    raw_cols = list(d.columns)
    d['_time'] = pd.to_datetime(d['transaction_datetime'], errors='coerce')
    d['_y'] = pd.to_numeric(d['is_fraud'], errors='coerce').fillna(0).astype(int)
    d['_card'] = d['card_id'].astype(str)
    d['_amount'] = d['transaction_amount'].astype(float)
    d['_category'] = d['merchant_category'].astype(str)
    d['age'] = pd.to_numeric(d['customer_age'], errors='coerce')
    d['gender_flag'] = d['customer_gender'].astype(str).str.strip().str.upper().str.startswith('M').astype(int)
    d = d.dropna(subset=['_time'])
    # identifiers, the time stamp, the label and the two columns that describe the fraud itself
    skip = {'transaction_id', 'customer_id', 'card_id', 'merchant_id', 'transaction_datetime',
            'is_fraud', 'fraud_type', 'fraud_amount_lost'}
    native = [c for c in raw_cols if c not in skip]
    text = {'merchant_category', 'merchant_country', 'transaction_day_of_week', 'customer_gender',
            'customer_state', 'transaction_channel', 'device_type', 'browser', 'email_domain'}
    return d, native, text

LOADERS = {'sparkov': load_sparkov, 'tabformer': load_tabformer, 'cc2026': load_cc2026}

In [6]:
# 3.1 Time gap, distance, velocity and location features
NIGHT_HOURS = [22, 23, 0, 1, 2, 3]

def build_features(d, ds):
    d = d.sort_values(['_card', '_time']).reset_index(drop=True)
    by_card = d.groupby('_card')
    d['_gap_h'] = by_card['_time'].diff().dt.total_seconds() / 3600

    if ds == 'sparkov':
        prev_lat, prev_lon = by_card['merch_lat'].shift(1), by_card['merch_long'].shift(1)
        d['_dist'] = haversine_distance(d['merch_lat'], d['merch_long'], prev_lat, prev_lon)
        d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 3600)          # +1 second so we never divide by zero
        d['dist_home_log'] = np.log1p(haversine_distance(d['lat'], d['long'], d['merch_lat'], d['merch_long']))
        LOCATION = ['dist_home_log', 'dist_log']

    elif ds == 'tabformer':
        # no GPS points: distance proxy = 1 if the merchant city changed since the card's previous transaction
        city = d['Merchant City'].astype(str)
        prev_city = by_card['Merchant City'].shift(1)
        d['_dist'] = np.where(prev_city.isna(), np.nan, (city != prev_city.astype(str)).astype(float))
        d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 60)            # +1 minute (TabFormer's clock resolution)
        ms, has = d['Merchant State'], d['Merchant State'].notna()
        d['abroad'] = (has & (ms.astype(str).str.len() > 2)).astype(int)          # foreign merchants have a country name
        d['outside_home_state'] = (has & (ms.astype(str) != d['State'].astype(str))).astype(int)
        LOCATION = ['outside_home_state', 'abroad']
        d = d[d['_time'] >= TF_WINDOW_START].copy()                # gaps were computed on the full history first

    elif ds == 'cc2026':
        d['_dist'] = pd.to_numeric(d['distance_from_last_tx_km'], errors='coerce')
        d['_vel'] = d['_dist'] / (d['_gap_h'] + 1 / 3600)
        d['dist_home_log'] = np.log1p(pd.to_numeric(d['distance_from_home_km'], errors='coerce').clip(lower=0))
        LOCATION = ['dist_home_log', 'dist_log', 'international_transaction', 'high_risk_country', 'merchant_country_risk']

    d['gap_log'] = np.log1p(d['_gap_h']).fillna(-1)                # -1 = the card's first transaction
    d['dist_log'] = np.log1p(d['_dist'].clip(lower=0)).fillna(-1)
    d['velocity_log'] = np.log1p(d['_vel'].clip(lower=0)).fillna(-1)
    d['amt_log'] = np.log1p(d['_amount'].clip(lower=0)).fillna(0)
    d['is_night'] = d['_time'].dt.hour.isin(NIGHT_HOURS).astype(int)
    d['age'] = d['age'].fillna(-1)
    d['day'] = d['_time'].dt.floor('D')
    d = d.sort_values('_time').reset_index(drop=True)
    return d, LOCATION

def gap_check(d, ds):
    check = pd.DataFrame({'fraud': d.loc[d['_y'] == 1, ['_gap_h', '_dist']].median(),
                          'normal': d.loc[d['_y'] == 0, ['_gap_h', '_dist']].median()})
    check.index = ['median time gap (hours)', 'median distance' + (' (city changed share)' if ds == 'tabformer' else ' (km)')]
    return check

In [7]:
# 4.1 Train / validation / test
def split_by_time(d, ds):
    if ds == 'sparkov':
        train, test = d[d['_split'] == 'train'], d[d['_split'] == 'test'].copy()
    else:
        cutoff_test = d['_time'].min() + 0.8 * (d['_time'].max() - d['_time'].min())
        train, test = d[d['_time'] < cutoff_test], d[d['_time'] >= cutoff_test].copy()
    train_sub, val, _ = chronological_val_split(train, '_time', val_frac=0.2)
    for name, part in [('train_sub', train_sub), ('val', val), ('test', test)]:
        print(f"{name:9s} {part['_time'].min()} to {part['_time'].max()}  rows={len(part):>9,}  "
              f"fraud={part['_y'].sum():>6,}  rate={part['_y'].mean():.4%}")
    return train_sub, val, test

In [8]:
# 5.1 category_risk and the native-column encodings
def as_number(s):
    return pd.to_numeric(s.astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')

def encode(train_sub, val, test, ds, NATIVE_COLS, TEXT_COLS):
    parts = (train_sub, val, test)
    risk_map = fit_category_risk(train_sub, '_category', '_y')
    for p in parts:
        p['category_risk'] = apply_category_risk(p, '_category', risk_map)

    if ds == 'cc2026':
        country_map = fit_category_risk(train_sub, 'merchant_country', '_y')
        for p in parts:
            p['merchant_country_risk'] = apply_category_risk(p, 'merchant_country', country_map)

    NATIVE_FEATURES, native_info = [], []
    y_fit, prior = train_sub['_y'], train_sub['_y'].mean()
    for col in NATIVE_COLS:
        name, s = f'raw__{col}', train_sub[col]
        if col not in TEXT_COLS and pd.api.types.is_bool_dtype(s):
            for p in parts: p[name] = p[col].astype(float)
            used = 'numeric'
        elif col not in TEXT_COLS and pd.api.types.is_numeric_dtype(s):
            fill = s.median()
            for p in parts: p[name] = p[col].astype(float).fillna(fill)
            used = 'numeric'
        elif col not in TEXT_COLS and as_number(s).notna().mean() > 0.95:       # e.g. '$1,234'
            fill = as_number(s).median()
            for p in parts: p[name] = as_number(p[col]).fillna(fill)
            used = 'numeric (parsed text)'
        elif s.astype(str).nunique() <= 1000:
            rates = y_fit.groupby(s.astype(str)).mean()
            for p in parts: p[name] = p[col].astype(str).map(rates).fillna(prior).astype(float)
            used = 'fraud rate of each value'
        else:
            native_info.append({'column': col, 'used as': 'left out (more than 1,000 values)'})
            continue
        NATIVE_FEATURES.append(name)
        native_info.append({'column': col, 'used as': used})
    print(f'all-native run uses {len(NATIVE_FEATURES)} columns')
    return NATIVE_FEATURES, pd.DataFrame(native_info)

In [9]:
# 6.1 Feature groups, training and the day-block bootstrap
SIX = ['amt_log', 'category_risk', 'is_night', 'gap_log', 'age', 'gender_flag']

def feature_groups(LOCATION, NATIVE_FEATURES):
    return {
        'Distance only':      ['dist_log'],
        'Time gap only':      ['gap_log'],
        'Velocity only':      ['velocity_log'],
        'Location only':      LOCATION,
        'Our six features':   SIX,
        'Six + location':     SIX + [c for c in LOCATION if c not in SIX],
        'All native columns': NATIVE_FEATURES,
    }

def make_rf(n_features):
    return lambda s: RandomForestClassifier(n_estimators=N_TREES, max_features=min(3, n_features),
                                            min_samples_leaf=5, random_state=s, n_jobs=-1)

def train_and_score(train_sub, val, test, ds, GROUPS):
    y_tr, y_val, y_test = train_sub['_y'], val['_y'], test['_y']
    prevalence = y_test.mean()
    test_days = test['day'].values
    rows = []
    for g, cols in GROUPS.items():
        seed_runs = run_seeds(f'{ds} {g}', make_rf(len(cols)),
                              train_sub[cols], y_tr, val[cols], y_val, test[cols], SEEDS,
                              checkpoint_dir=checkpoint_dir(ds))
        pairs = [(r['test_proba'], r['threshold']) for r in seed_runs]
        boot = bootstrap_model(y_test, pairs, test_days, n_boot=N_BOOT, boot_seed=BOOT_SEED)
        s = summarise(y_test, pairs, boot)
        rows.append({'Dataset': ds, 'Feature group': g, 'Features': len(cols),
                     'Seeds': len(SEEDS), 'Test rows': len(y_test), 'Test fraud': int(y_test.sum()),
                     'Prevalence': prevalence,
                     'Val PR-AUC': np.mean([r['val_pr_auc'] for r in seed_runs]),
                     'PR-AUC': s['PR-AUC'], 'PR-AUC CI low': s['PR-AUC CI low'], 'PR-AUC CI high': s['PR-AUC CI high'],
                     'Lift': s['PR-AUC'] / prevalence,
                     'Lift CI low': s['PR-AUC CI low'] / prevalence, 'Lift CI high': s['PR-AUC CI high'] / prevalence,
                     'F2': s['F2'], 'F2 CI low': s['F2 CI low'], 'F2 CI high': s['F2 CI high'],
                     'MCC': s['MCC'], 'MCC CI low': s['MCC CI low'], 'MCC CI high': s['MCC CI high'],
                     'Thresholds': s['Thresholds']})
        print(f'{g}: done')
    return pd.DataFrame(rows)

def results_table(result):
    return pd.DataFrame({
        'PR-AUC [95% CI]': [f"{r['PR-AUC']:.4f} [{r['PR-AUC CI low']:.4f}, {r['PR-AUC CI high']:.4f}]" for _, r in result.iterrows()],
        'Lift [95% CI]':   [f"{r['Lift']:.1f} [{r['Lift CI low']:.1f}, {r['Lift CI high']:.1f}]" for _, r in result.iterrows()],
        'F2 [95% CI]':     [f"{r['F2']:.4f} [{r['F2 CI low']:.4f}, {r['F2 CI high']:.4f}]" for _, r in result.iterrows()],
        'MCC [95% CI]':    [f"{r['MCC']:.4f} [{r['MCC CI low']:.4f}, {r['MCC CI high']:.4f}]" for _, r in result.iterrows()],
    }, index=result['Feature group'])

## 3. Loading the saved predictions
`run_seeds` finds each finished seed in the Drive folder of `cross_dataset_protocol.ipynb` and loads its test predictions and threshold. We check that each loaded seed has one prediction for every test row.

In [10]:
# 3.1 The saved predictions of every feature group
def load_runs(train_sub, val, test, ds, GROUPS):
    runs = {}
    for g, cols in GROUPS.items():
        runs[g] = run_seeds(f'{ds} {g}', make_rf(len(cols)),
                            train_sub[cols], train_sub['_y'], val[cols], val['_y'], test[cols], SEEDS,
                            checkpoint_dir=checkpoint_dir(ds))
        assert all(len(r['test_proba']) == len(test) for r in runs[g]), f'{g}: predictions do not match the test rows'
    return runs

## 4. Recall, dollar recall and precision at each budget
For each seed we sort the test transactions from the highest score to the lowest and flag the top share. In each bootstrap resample, a day drawn twice counts twice and a day not drawn counts zero times, exactly as in `bootstrap_model`, and the budget is a share of the resampled transactions. We draw the resampled days in the same way as `bootstrap_model`, so the alert budgets and Table 1 use the same resamples. Each seed's test rows are sorted once, which makes the bootstrap much faster without changing any number.

In [11]:
# 4.1 Point estimates and day-block bootstrap intervals
BUDGETS = [0.005, 0.01, 0.02]
METRICS = ['Recall', 'Dollar recall', 'Precision']

def budget_scores(w, y, a, budget):
    # w, y, a: row weights, labels and amounts, sorted from the highest score to the lowest
    flagged = np.cumsum(w) <= budget * w.sum()
    caught = w * y * flagged
    return {'Recall': caught.sum() / (w * y).sum(),
            'Dollar recall': (caught * a).sum() / max((w * y * a).sum(), 1e-12),
            'Precision': caught.sum() / max((w * flagged).sum(), 1e-12)}

def alert_budgets(test, runs, ds):
    y_arr = test['_y'].values.astype(float)
    amt_arr = test['_amount'].clip(lower=0).fillna(0).values      # refunds and negative amounts count as 0
    days = pd.Series(test['day'].values)
    unique_days = days.unique()
    day_index = pd.Index(unique_days)
    day_of_row = day_index.get_indexer(days)
    print(f'test transactions: {len(y_arr):,} | fraud: {int(y_arr.sum()):,} ({y_arr.mean():.3%}) | '
          f'fraud dollars: ${amt_arr[y_arr == 1].sum():,.0f} | alerts: '
          + ', '.join(f'{b:.1%} = {int(b * len(y_arr)):,}' for b in BUDGETS))

    # sort each seed's test rows once
    sorted_runs = {g: [(o, y_arr[o], amt_arr[o], day_of_row[o])
                       for o in (np.argsort(-r['test_proba'], kind='stable') for r in seed_runs)]
                   for g, seed_runs in runs.items()}

    def scores_for(times_drawn):
        out = {}
        for g, seeds in sorted_runs.items():
            per_budget = {b: [] for b in BUDGETS}
            for o, y_s, a_s, day_s in seeds:
                w_s = np.ones(len(o)) if times_drawn is None else times_drawn[day_s]
                for b in BUDGETS:
                    per_budget[b].append(budget_scores(w_s, y_s, a_s, b))
            for b in BUDGETS:
                out[(g, b)] = {m: np.mean([s[m] for s in per_budget[b]]) for m in METRICS}
        return out

    point = scores_for(None)
    rng = np.random.default_rng(BOOT_SEED)      # the same resampled days as bootstrap_model
    boot = []
    for _ in range(N_BOOT):
        drawn = rng.choice(unique_days, size=len(unique_days), replace=True)
        times_drawn = np.bincount(day_index.get_indexer(drawn), minlength=len(unique_days)).astype(float)
        boot.append(scores_for(times_drawn))

    rows = []
    for (g, b), p in point.items():
        row = {'Dataset': ds, 'Feature group': g, 'Budget': b, 'Alerts': int(b * len(y_arr)),
               'Seeds': len(SEEDS), 'Prevalence': y_arr.mean()}
        for m in METRICS:
            vals = np.array([d[(g, b)][m] for d in boot])
            row.update({m: p[m], f'{m} CI low': np.percentile(vals, 2.5), f'{m} CI high': np.percentile(vals, 97.5)})
        rows.append(row)
    return pd.DataFrame(rows)

def fmt(r, m):
    return f"{r[m]:.3f} [{r[m + ' CI low']:.3f}, {r[m + ' CI high']:.3f}]"

def budget_table(result):
    return pd.DataFrame({'Alerts': result['Alerts'].values,
                         'Recall [95% CI]': [fmt(r, 'Recall') for _, r in result.iterrows()],
                         'Dollar recall [95% CI]': [fmt(r, 'Dollar recall') for _, r in result.iterrows()],
                         'Precision [95% CI]': [fmt(r, 'Precision') for _, r in result.iterrows()]},
                        index=pd.MultiIndex.from_arrays([result['Feature group'], result['Budget'].map('{:.1%}'.format)],
                                                        names=['Feature group', 'Budget']))

## 5. A check against the earlier run
The earlier version of this notebook scored CC Fraud 2026 with a slower loop. If its saved results are in Google Drive, we compare them with the new ones, which should match exactly.

In [12]:
# 5.1 Compare with the saved CC Fraud 2026 result, if there is one
def compare_with_saved(result, ds):
    path = f"{RESULTS_DIR}/point3_{ds}{'_quick' if QUICK else ''}.csv"
    if not os.path.exists(path):
        return None
    saved = pd.read_csv(path)
    cols = [c for c in saved.columns if c.startswith(tuple(METRICS))]
    merged = saved.merge(result, on=['Feature group', 'Budget'], suffixes=('_saved', '_new'))
    worst = max(abs(merged[f'{c}_saved'] - merged[f'{c}_new']).max() for c in cols)
    print(f'{ds}: largest difference from the saved result = {worst:.2e}')
    return worst

## 6. Running the three datasets
For each dataset we load the data, build the features, split by time, load the saved predictions and measure the alert budgets. Each dataset's results are saved to Google Drive before the next dataset starts.

In [13]:
# 6.1 One dataset after another
os.makedirs('results', exist_ok=True)
for ds in DATASETS:
    print('=' * 100, f'\n{NAMES[ds]}\n' + '=' * 100)
    d, NATIVE_COLS, TEXT_COLS = LOADERS[ds]()
    d, LOCATION = build_features(d, ds)
    train_sub, val, test = split_by_time(d, ds)
    del d; gc.collect()
    NATIVE_FEATURES, _ = encode(train_sub, val, test, ds, NATIVE_COLS, TEXT_COLS)
    GROUPS = feature_groups(LOCATION, NATIVE_FEATURES)

    runs = load_runs(train_sub, val, test, ds, GROUPS)
    result = alert_budgets(test, runs, ds)
    compare_with_saved(result, ds)
    out = f"{RESULTS_DIR}/point3_{ds}{'_quick' if QUICK else ''}.csv"
    result.to_csv(out, index=False)
    result.to_csv(f"results/point3_{ds}{'_quick' if QUICK else ''}.csv", index=False)
    print('saved ->', out)
    display(budget_table(result))
    one = result[result['Budget'] == 0.01].set_index('Feature group')
    print(f'{NAMES[ds]}: recall and dollar recall at the 1% budget')
    display(pd.DataFrame({'Recall at 1% [95% CI]': [fmt(r, 'Recall') for _, r in one.iterrows()],
                          'Dollar recall at 1% [95% CI]': [fmt(r, 'Dollar recall') for _, r in one.iterrows()]},
                         index=one.index))
    del train_sub, val, test, runs; gc.collect()

CC Fraud 2026
train_sub 2026-01-01 02:28:51 to 2026-08-22 14:57:50  rows=   63,796  fraud= 1,293  rate=2.0268%
val       2026-08-22 15:08:05 to 2026-10-20 00:08:49  rows=   16,186  fraud=   344  rate=2.1253%
test      2026-10-20 00:24:28 to 2026-12-31 23:45:47  rows=   20,018  fraud=   418  rate=2.0881%
all-native run uses 35 columns
cc2026 Distance only seed 42: loaded from checkpoint
cc2026 Distance only seed 43: loaded from checkpoint
cc2026 Distance only seed 44: loaded from checkpoint
cc2026 Distance only seed 45: loaded from checkpoint
cc2026 Distance only seed 46: loaded from checkpoint
cc2026 Time gap only seed 42: loaded from checkpoint
cc2026 Time gap only seed 43: loaded from checkpoint
cc2026 Time gap only seed 44: loaded from checkpoint
cc2026 Time gap only seed 45: loaded from checkpoint
cc2026 Time gap only seed 46: loaded from checkpoint
cc2026 Velocity only seed 42: loaded from checkpoint
cc2026 Velocity only seed 43: loaded from checkpoint
cc2026 Velocity only seed 44

Alerts       Recall [95% CI]  \
Feature group      Budget                                 
Distance only      0.5%       100  0.006 [0.000, 0.014]   
                   1.0%       200  0.015 [0.005, 0.024]   
                   2.0%       400  0.026 [0.012, 0.042]   
Time gap only      0.5%       100  0.006 [0.000, 0.013]   
                   1.0%       200  0.011 [0.002, 0.021]   
                   2.0%       400  0.022 [0.010, 0.038]   
Velocity only      0.5%       100  0.007 [0.000, 0.015]   
                   1.0%       200  0.009 [0.002, 0.018]   
                   2.0%       400  0.019 [0.008, 0.032]   
Location only      0.5%       100  0.030 [0.014, 0.046]   
                   1.0%       200  0.044 [0.026, 0.065]   
                   2.0%       400  0.079 [0.053, 0.103]   
Our six features   0.5%       100  0.003 [0.000, 0.008]   
                   1.0%       200  0.005 [0.000, 0.012]   
                   2.0%       400  0.020 [0.011, 0.032]   
Six + location     0.5%       100  0.028 [0.015, 0.043]   
                   1.0%       200  0.056 [0.036, 0.075]   
                   2.0%       400  0.119 [0.095, 0.144]   
All native columns 0.5%       100  0.072 [0.054, 0.088]   
                   1.0%       200  0.130 [0.106, 0.149]   
                   2.0%       400  0.216 [0.187, 0.242]   

                          Dollar recall [95% CI]    Precision [95% CI]  
Feature group      Budget                                               
Distance only      0.5%     0.005 [0.000, 0.014]  0.024 [0.001, 0.059]  
                   1.0%     0.010 [0.001, 0.019]  0.031 [0.011, 0.053]  
                   2.0%     0.016 [0.006, 0.029]  0.028 [0.013, 0.044]  
Time gap only      0.5%     0.001 [0.000, 0.002]  0.024 [0.000, 0.055]  
                   1.0%     0.001 [0.000, 0.004]  0.022 [0.005, 0.045]  
                   2.0%     0.009 [0.001, 0.024]  0.024 [0.010, 0.040]  
Velocity only      0.5%     0.001 [0.000, 0.002]  0.028 [0.000, 0.067]  
                   1.0%     0.001 [0.000, 0.002]  0.019 [0.005, 0.039]  
                   2.0%     0.006 [0.001, 0.017]  0.020 [0.009, 0.033]  
Location only      0.5%     0.019 [0.007, 0.035]  0.126 [0.062, 0.197]  
                   1.0%     0.030 [0.011, 0.058]  0.092 [0.055, 0.138]  
                   2.0%     0.051 [0.026, 0.082]  0.083 [0.055, 0.113]  
Our six features   0.5%     0.007 [0.000, 0.019]  0.014 [0.000, 0.034]  
                   1.0%     0.014 [0.000, 0.041]  0.011 [0.001, 0.026]  
                   2.0%     0.044 [0.015, 0.079]  0.021 [0.011, 0.034]  
Six + location     0.5%     0.077 [0.015, 0.173]  0.116 [0.065, 0.189]  
                   1.0%     0.113 [0.041, 0.210]  0.116 [0.076, 0.159]  
                   2.0%     0.170 [0.088, 0.271]  0.125 [0.097, 0.153]  
All native columns 0.5%     0.125 [0.050, 0.219]  0.300 [0.227, 0.381]  
                   1.0%     0.237 [0.150, 0.328]  0.271 [0.217, 0.322]  
                   2.0%     0.376 [0.270, 0.460]  0.226 [0.191, 0.265]

CC Fraud 2026: recall and dollar recall at the 1% budget


,Recall at 1% [95% CI],Dollar recall at 1% [95% CI]
Feature group,,
Distance only,"0.015 [0.005, 0.024]","0.010 [0.001, 0.019]"
Time gap only,"0.011 [0.002, 0.021]","0.001 [0.000, 0.004]"
Velocity only,"0.009 [0.002, 0.018]","0.001 [0.000, 0.002]"
Location only,"0.044 [0.026, 0.065]","0.030 [0.011, 0.058]"
Our six features,"0.005 [0.000, 0.012]","0.014 [0.000, 0.041]"
Six + location,"0.056 [0.036, 0.075]","0.113 [0.041, 0.210]"
All native columns,"0.130 [0.106, 0.149]","0.237 [0.150, 0.328]"


Sparkov
train_sub 2019-01-01 00:00:18 to 2020-03-06 00:06:49  rows=1,036,901  fraud= 5,968  rate=0.5756%
val       2020-03-06 00:12:04 to 2020-06-21 12:13:37  rows=  259,774  fraud= 1,538  rate=0.5921%
test      2020-06-21 12:14:25 to 2020-12-31 23:59:34  rows=  555,719  fraud= 2,145  rate=0.3860%
all-native run uses 13 columns
sparkov Distance only seed 42: loaded from checkpoint
sparkov Distance only seed 43: loaded from checkpoint
sparkov Distance only seed 44: loaded from checkpoint
sparkov Distance only seed 45: loaded from checkpoint
sparkov Distance only seed 46: loaded from checkpoint
sparkov Time gap only seed 42: loaded from checkpoint
sparkov Time gap only seed 43: loaded from checkpoint
sparkov Time gap only seed 44: loaded from checkpoint
sparkov Time gap only seed 45: loaded from checkpoint
sparkov Time gap only seed 46: loaded from checkpoint
sparkov Velocity only seed 42: loaded from checkpoint
sparkov Velocity only seed 43: loaded from checkpoint
sparkov Velocity only 

Alerts       Recall [95% CI]  \
Feature group      Budget                                 
Distance only      0.5%      2778  0.004 [0.002, 0.007]   
                   1.0%      5557  0.010 [0.006, 0.014]   
                   2.0%     11114  0.021 [0.015, 0.026]   
Time gap only      0.5%      2778  0.008 [0.005, 0.013]   
                   1.0%      5557  0.012 [0.008, 0.018]   
                   2.0%     11114  0.026 [0.020, 0.033]   
Velocity only      0.5%      2778  0.007 [0.004, 0.010]   
                   1.0%      5557  0.015 [0.010, 0.019]   
                   2.0%     11114  0.026 [0.020, 0.032]   
Location only      0.5%      2778  0.009 [0.005, 0.012]   
                   1.0%      5557  0.013 [0.009, 0.018]   
                   2.0%     11114  0.024 [0.019, 0.030]   
Our six features   0.5%      2778  0.843 [0.819, 0.865]   
                   1.0%      5557  0.921 [0.906, 0.932]   
                   2.0%     11114  0.953 [0.941, 0.963]   
Six + location     0.5%      2778  0.836 [0.809, 0.858]   
                   1.0%      5557  0.911 [0.897, 0.925]   
                   2.0%     11114  0.948 [0.936, 0.959]   
All native columns 0.5%      2778  0.337 [0.311, 0.364]   
                   1.0%      5557  0.404 [0.377, 0.432]   
                   2.0%     11114  0.484 [0.454, 0.512]   

                          Dollar recall [95% CI]    Precision [95% CI]  
Feature group      Budget                                               
Distance only      0.5%     0.004 [0.002, 0.008]  0.003 [0.002, 0.005]  
                   1.0%     0.009 [0.005, 0.014]  0.004 [0.002, 0.005]  
                   2.0%     0.020 [0.013, 0.027]  0.004 [0.003, 0.005]  
Time gap only      0.5%     0.010 [0.006, 0.016]  0.006 [0.004, 0.009]  
                   1.0%     0.015 [0.009, 0.022]  0.005 [0.003, 0.007]  
                   2.0%     0.031 [0.023, 0.041]  0.005 [0.004, 0.006]  
Velocity only      0.5%     0.006 [0.003, 0.009]  0.005 [0.003, 0.008]  
                   1.0%     0.013 [0.008, 0.018]  0.006 [0.004, 0.008]  
                   2.0%     0.025 [0.018, 0.033]  0.005 [0.004, 0.006]  
Location only      0.5%     0.009 [0.004, 0.013]  0.007 [0.004, 0.010]  
                   1.0%     0.013 [0.008, 0.018]  0.005 [0.003, 0.007]  
                   2.0%     0.024 [0.017, 0.032]  0.005 [0.003, 0.006]  
Our six features   0.5%     0.944 [0.930, 0.957]  0.651 [0.586, 0.722]  
                   1.0%     0.981 [0.975, 0.986]  0.355 [0.315, 0.403]  
                   2.0%     0.994 [0.991, 0.996]  0.184 [0.163, 0.208]  
Six + location     0.5%     0.946 [0.930, 0.958]  0.646 [0.579, 0.712]  
                   1.0%     0.981 [0.975, 0.985]  0.352 [0.312, 0.398]  
                   2.0%     0.993 [0.990, 0.995]  0.183 [0.162, 0.207]  
All native columns 0.5%     0.285 [0.246, 0.321]  0.260 [0.226, 0.297]  
                   1.0%     0.336 [0.295, 0.376]  0.156 [0.135, 0.179]  
                   2.0%     0.447 [0.407, 0.485]  0.093 [0.082, 0.106]

... (2 outputs left out)

TabFormer
train_sub 2015-01-01 00:01:00 to 2018-04-21 09:28:00  rows=1,168,504  fraud= 1,511  rate=0.1293%
val       2018-04-21 09:29:00 to 2019-02-16 23:50:00  rows=  293,511  fraud=   362  rate=0.1233%
test      2019-02-17 00:00:00 to 2020-02-28 23:49:00  rows=  379,534  fraud=   309  rate=0.0814%
all-native run uses 18 columns
tabformer Distance only seed 42: loaded from checkpoint
tabformer Distance only seed 43: loaded from checkpoint
tabformer Distance only seed 44: loaded from checkpoint
tabformer Distance only seed 45: loaded from checkpoint
tabformer Distance only seed 46: loaded from checkpoint
tabformer Time gap only seed 42: loaded from checkpoint
tabformer Time gap only seed 43: loaded from checkpoint
tabformer Time gap only seed 44: loaded from checkpoint
tabformer Time gap only seed 45: loaded from checkpoint
tabformer Time gap only seed 46: loaded from checkpoint
tabformer Velocity only seed 42: loaded from checkpoint
tabformer Velocity only seed 43: loaded from checkpo

Alerts       Recall [95% CI]  \
Feature group      Budget                                 
Distance only      0.5%      1897  0.000 [0.000, 0.012]   
                   1.0%      3795  0.010 [0.000, 0.025]   
                   2.0%      7590  0.019 [0.003, 0.041]   
Time gap only      0.5%      1897  0.003 [0.000, 0.011]   
                   1.0%      3795  0.011 [0.001, 0.023]   
                   2.0%      7590  0.025 [0.011, 0.043]   
Velocity only      0.5%      1897  0.009 [0.000, 0.021]   
                   1.0%      3795  0.016 [0.004, 0.032]   
                   2.0%      7590  0.028 [0.013, 0.047]   
Location only      0.5%      1897  1.000 [0.967, 1.000]   
                   1.0%      3795  1.000 [1.000, 1.000]   
                   2.0%      7590  1.000 [1.000, 1.000]   
Our six features   0.5%      1897  0.141 [0.108, 0.174]   
                   1.0%      3795  0.179 [0.142, 0.214]   
                   2.0%      7590  0.241 [0.195, 0.288]   
Six + location     0.5%      1897  0.682 [0.639, 0.727]   
                   1.0%      3795  0.770 [0.727, 0.809]   
                   2.0%      7590  0.841 [0.805, 0.877]   
All native columns 0.5%      1897  1.000 [1.000, 1.000]   
                   1.0%      3795  1.000 [1.000, 1.000]   
                   2.0%      7590  1.000 [1.000, 1.000]   

                          Dollar recall [95% CI]    Precision [95% CI]  
Feature group      Budget                                               
Distance only      0.5%     0.000 [0.000, 0.065]  0.000 [0.000, 0.002]  
                   1.0%     0.055 [0.000, 0.127]  0.001 [0.000, 0.002]  
                   2.0%     0.056 [0.000, 0.150]  0.001 [0.000, 0.002]  
Time gap only      0.5%     0.001 [0.000, 0.003]  0.001 [0.000, 0.002]  
                   1.0%     0.014 [0.000, 0.036]  0.001 [0.000, 0.002]  
                   2.0%     0.027 [0.006, 0.060]  0.001 [0.000, 0.002]  
Velocity only      0.5%     0.003 [0.000, 0.008]  0.001 [0.000, 0.003]  
                   1.0%     0.006 [0.001, 0.014]  0.001 [0.000, 0.003]  
                   2.0%     0.023 [0.004, 0.049]  0.001 [0.001, 0.002]  
Location only      0.5%     1.000 [0.957, 1.000]  0.163 [0.131, 0.195]  
                   1.0%     1.000 [1.000, 1.000]  0.081 [0.065, 0.100]  
                   2.0%     1.000 [1.000, 1.000]  0.041 [0.033, 0.050]  
Our six features   0.5%     0.282 [0.181, 0.375]  0.023 [0.016, 0.031]  
                   1.0%     0.337 [0.223, 0.435]  0.015 [0.011, 0.019]  
                   2.0%     0.420 [0.305, 0.516]  0.010 [0.007, 0.013]  
Six + location     0.5%     0.748 [0.671, 0.819]  0.111 [0.091, 0.136]  
                   1.0%     0.838 [0.772, 0.890]  0.063 [0.051, 0.076]  
                   2.0%     0.912 [0.872, 0.945]  0.034 [0.028, 0.042]  
All native columns 0.5%     1.000 [1.000, 1.000]  0.163 [0.131, 0.200]  
                   1.0%     1.000 [1.000, 1.000]  0.081 [0.065, 0.100]  
                   2.0%     1.000 [1.000, 1.000]  0.041 [0.033, 0.050]

TabFormer: recall and dollar recall at the 1% budget


,Recall at 1% [95% CI],Dollar recall at 1% [95% CI]
Feature group,,
Distance only,"0.010 [0.000, 0.025]","0.055 [0.000, 0.127]"
Time gap only,"0.011 [0.001, 0.023]","0.014 [0.000, 0.036]"
Velocity only,"0.016 [0.004, 0.032]","0.006 [0.001, 0.014]"
Location only,"1.000 [1.000, 1.000]","1.000 [1.000, 1.000]"
Our six features,"0.179 [0.142, 0.214]","0.337 [0.223, 0.435]"
Six + location,"0.770 [0.727, 0.809]","0.838 [0.772, 0.890]"
All native columns,"1.000 [1.000, 1.000]","1.000 [1.000, 1.000]"


## 7. Table 3: every finished dataset together
We read the saved results of every finished dataset. Table 3a gives the recall and Table 3b the dollar recall, with datasets and budgets as rows and feature groups as columns. If a dataset has both a quick and a full run, we keep the full run.

In [14]:
# 7.1 Combine the datasets
ORDER_G = list(GROUPS)
ORDER_D = ['sparkov', 'tabformer', 'cc2026']

files = sorted(glob.glob(f'{RESULTS_DIR}/point3_*.csv'))
print('found:', [os.path.basename(f) for f in files])
allres = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
allres = (allres.sort_values('Seeds', ascending=False)
                .drop_duplicates(['Dataset', 'Feature group', 'Budget']).reset_index(drop=True))
allres['Dataset'] = pd.Categorical(allres['Dataset'].map(NAMES),
                                   [NAMES[x] for x in ORDER_D if x in set(allres['Dataset'])], ordered=True)
allres['Budget'] = allres['Budget'].map('{:.1%}'.format)

def table(metric):
    return (allres.assign(v=[fmt(r, metric) for _, r in allres.iterrows()])
                  .pivot_table(index=['Dataset', 'Budget'], columns='Feature group', values='v',
                               aggfunc='first', observed=True)
                  .reindex(columns=ORDER_G))

table3_recall, table3_dollar = table('Recall'), table('Dollar recall')
table3_recall.to_csv(f'{RESULTS_DIR}/table3_recall.csv')
table3_dollar.to_csv(f'{RESULTS_DIR}/table3_dollar_recall.csv')
print('Table 3a: recall at each alert budget [95% CI]'); display(table3_recall)
print('Table 3b: dollar recall at each alert budget [95% CI]'); display(table3_dollar)

found: ['point3_cc2026.csv', 'point3_sparkov.csv', 'point3_tabformer.csv']
Table 3a: recall at each alert budget [95% CI]


Feature group                Distance only         Time gap only  \
Dataset       Budget                                               
Sparkov       0.5%    0.004 [0.002, 0.007]  0.008 [0.005, 0.013]   
              1.0%    0.010 [0.006, 0.014]  0.012 [0.008, 0.018]   
              2.0%    0.021 [0.015, 0.026]  0.026 [0.020, 0.033]   
TabFormer     0.5%    0.000 [0.000, 0.012]  0.003 [0.000, 0.011]   
              1.0%    0.010 [0.000, 0.025]  0.011 [0.001, 0.023]   
              2.0%    0.019 [0.003, 0.041]  0.025 [0.011, 0.043]   
CC Fraud 2026 0.5%    0.006 [0.000, 0.014]  0.006 [0.000, 0.013]   
              1.0%    0.015 [0.005, 0.024]  0.011 [0.002, 0.021]   
              2.0%    0.026 [0.012, 0.042]  0.022 [0.010, 0.038]   

Feature group                Velocity only         Location only  \
Dataset       Budget                                               
Sparkov       0.5%    0.007 [0.004, 0.010]  0.009 [0.005, 0.012]   
              1.0%    0.015 [0.010, 0.019]  0.013 [0.009, 0.018]   
              2.0%    0.026 [0.020, 0.032]  0.024 [0.019, 0.030]   
TabFormer     0.5%    0.009 [0.000, 0.021]  1.000 [0.967, 1.000]   
              1.0%    0.016 [0.004, 0.032]  1.000 [1.000, 1.000]   
              2.0%    0.028 [0.013, 0.047]  1.000 [1.000, 1.000]   
CC Fraud 2026 0.5%    0.007 [0.000, 0.015]  0.030 [0.014, 0.046]   
              1.0%    0.009 [0.002, 0.018]  0.044 [0.026, 0.065]   
              2.0%    0.019 [0.008, 0.032]  0.079 [0.053, 0.103]   

Feature group             Our six features        Six + location  \
Dataset       Budget                                               
Sparkov       0.5%    0.843 [0.819, 0.865]  0.836 [0.809, 0.858]   
              1.0%    0.921 [0.906, 0.932]  0.911 [0.897, 0.925]   
              2.0%    0.953 [0.941, 0.963]  0.948 [0.936, 0.959]   
TabFormer     0.5%    0.141 [0.108, 0.174]  0.682 [0.639, 0.727]   
              1.0%    0.179 [0.142, 0.214]  0.770 [0.727, 0.809]   
              2.0%    0.241 [0.195, 0.288]  0.841 [0.805, 0.877]   
CC Fraud 2026 0.5%    0.003 [0.000, 0.008]  0.028 [0.015, 0.043]   
              1.0%    0.005 [0.000, 0.012]  0.056 [0.036, 0.075]   
              2.0%    0.020 [0.011, 0.032]  0.119 [0.095, 0.144]   

Feature group           All native columns  
Dataset       Budget                        
Sparkov       0.5%    0.337 [0.311, 0.364]  
              1.0%    0.404 [0.377, 0.432]  
              2.0%    0.484 [0.454, 0.512]  
TabFormer     0.5%    1.000 [1.000, 1.000]  
              1.0%    1.000 [1.000, 1.000]  
              2.0%    1.000 [1.000, 1.000]  
CC Fraud 2026 0.5%    0.072 [0.054, 0.088]  
              1.0%    0.130 [0.106, 0.149]  
              2.0%    0.216 [0.187, 0.242]

Table 3b: dollar recall at each alert budget [95% CI]


Feature group                Distance only         Time gap only  \
Dataset       Budget                                               
Sparkov       0.5%    0.004 [0.002, 0.008]  0.010 [0.006, 0.016]   
              1.0%    0.009 [0.005, 0.014]  0.015 [0.009, 0.022]   
              2.0%    0.020 [0.013, 0.027]  0.031 [0.023, 0.041]   
TabFormer     0.5%    0.000 [0.000, 0.065]  0.001 [0.000, 0.003]   
              1.0%    0.055 [0.000, 0.127]  0.014 [0.000, 0.036]   
              2.0%    0.056 [0.000, 0.150]  0.027 [0.006, 0.060]   
CC Fraud 2026 0.5%    0.005 [0.000, 0.014]  0.001 [0.000, 0.002]   
              1.0%    0.010 [0.001, 0.019]  0.001 [0.000, 0.004]   
              2.0%    0.016 [0.006, 0.029]  0.009 [0.001, 0.024]   

Feature group                Velocity only         Location only  \
Dataset       Budget                                               
Sparkov       0.5%    0.006 [0.003, 0.009]  0.009 [0.004, 0.013]   
              1.0%    0.013 [0.008, 0.018]  0.013 [0.008, 0.018]   
              2.0%    0.025 [0.018, 0.033]  0.024 [0.017, 0.032]   
TabFormer     0.5%    0.003 [0.000, 0.008]  1.000 [0.957, 1.000]   
              1.0%    0.006 [0.001, 0.014]  1.000 [1.000, 1.000]   
              2.0%    0.023 [0.004, 0.049]  1.000 [1.000, 1.000]   
CC Fraud 2026 0.5%    0.001 [0.000, 0.002]  0.019 [0.007, 0.035]   
              1.0%    0.001 [0.000, 0.002]  0.030 [0.011, 0.058]   
              2.0%    0.006 [0.001, 0.017]  0.051 [0.026, 0.082]   

Feature group             Our six features        Six + location  \
Dataset       Budget                                               
Sparkov       0.5%    0.944 [0.930, 0.957]  0.946 [0.930, 0.958]   
              1.0%    0.981 [0.975, 0.986]  0.981 [0.975, 0.985]   
              2.0%    0.994 [0.991, 0.996]  0.993 [0.990, 0.995]   
TabFormer     0.5%    0.282 [0.181, 0.375]  0.748 [0.671, 0.819]   
              1.0%    0.337 [0.223, 0.435]  0.838 [0.772, 0.890]   
              2.0%    0.420 [0.305, 0.516]  0.912 [0.872, 0.945]   
CC Fraud 2026 0.5%    0.007 [0.000, 0.019]  0.077 [0.015, 0.173]   
              1.0%    0.014 [0.000, 0.041]  0.113 [0.041, 0.210]   
              2.0%    0.044 [0.015, 0.079]  0.170 [0.088, 0.271]   

Feature group           All native columns  
Dataset       Budget                        
Sparkov       0.5%    0.285 [0.246, 0.321]  
              1.0%    0.336 [0.295, 0.376]  
              2.0%    0.447 [0.407, 0.485]  
TabFormer     0.5%    1.000 [1.000, 1.000]  
              1.0%    1.000 [1.000, 1.000]  
              2.0%    1.000 [1.000, 1.000]  
CC Fraud 2026 0.5%    0.125 [0.050, 0.219]  
              1.0%    0.237 [0.150, 0.328]  
              2.0%    0.376 [0.270, 0.460]

## 8. Discussion

### Credit Card Fraud 2026

**The test period.** The test set has 20,018 transactions, of which 418 are fraud (2.09%), worth $83,957. The three budgets mean reviewing 100, 200 and 400 transactions. A model that ranks at random would catch about the same share of fraud as the budget, so a recall of about 0.005, 0.01 and 0.02, and its precision would equal the fraud rate (about 0.021).

**The movement features and our six features catch no more than random review.** At the 1% budget, distance only, time gap only and velocity only catch between 0.009 and 0.015 of the fraud transactions, and our six features catch 0.005 [0.000, 0.012]. Their precision at 1% stays between 0.011 and 0.031, close to the fraud rate. Their dollar recall is even lower (0.001 to 0.014), so the few frauds they do flag are small ones. This matches Table 1, where all four groups have a lift close to 1.

**Location is what lets the model catch fraud.** Location only catches 0.044 [0.026, 0.065] of the fraud at 1%, about four times the random rate, with a precision of 0.092. Adding the location features to our six features raises the recall at 1% from 0.005 to 0.056 [0.036, 0.075] and at 2% from 0.020 to 0.119 [0.095, 0.144]; the intervals do not overlap. The precision of six + location stays between 0.116 and 0.125 at all three budgets, about six times the fraud rate.

**Location catches the larger frauds.** For six + location, the dollar recall is about double the recall (0.113 against 0.056 at 1%, and 0.170 against 0.119 at 2%). The fraud transactions it flags are larger than average, which matters more to a bank than the number of cases. The intervals for dollar recall are wide (for example [0.041, 0.210] at 1%) because a few large frauds carry much of the total.

**Even the dataset's own columns leave most fraud unflagged.** All 35 native columns catch 0.130 [0.106, 0.149] of the fraud transactions and 0.237 [0.150, 0.328] of the fraud dollars at 1%, and 0.216 and 0.376 at 2%. Their precision is 0.271 at 1%, so about one in four reviewed transactions is fraud. Still, after reviewing 2% of transactions, about four in five fraud transactions are missed. On Sparkov, our final model catches 0.920 of the fraud transactions and 0.981 of the fraud dollars at the 1% budget (`final_model_gap.ipynb`).

**What this means.** The alert budgets give the same answer as PR-AUC in Table 1: on CC 2026 the time gap does not help, and location is the only part of our features that a fraud team could use. Because even the upper bound misses most fraud, we keep CC 2026 as a secondary dataset.